TODOs:
* Deal with zero-inflated data.
* (DONE) Deal with non-nornally distributed featuers.
* Knowledge-based metric, read up paper's, blog posts etc.
* Combination of a knowledge-based and ML-based metric.
* Analyze data across leagues. If significantly different, could you the league as feature as well.
* Fix bug: unnormalized by minutes even though I normalized by number of player's possessions
* Undo Yeo transform

In [1]:
from pathlib import Path
import sys

NOTEBOOKS_DIR = Path.cwd().resolve().parent
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOKS_DIR))

from paths import CLUSTERS_DIR, INTERMEDIATE_DATA_DIR, REFERENCE_DATA_DIR, TRAINING_DATA_DIR, ensure_output_dirs

ensure_output_dirs()

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [33]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA   # might not be needed

csv_file = TRAINING_DATA_DIR / "train_data_yeo.csv"
data_full = pd.read_csv(csv_file)
# data_full_standardized = StandardScaler().fit_transform(data_full)
# keep test set for after model selection via cross-validation to evaluate the final model
# train_data, test_data = train_test_split(data_full_standardized, test_size=0.2, random_state=42)


In [34]:
OTHER_FEATURES = [
    "rebounds", "offensive rebounds", "defensive rebounds", "fouls", "foulds drawn", "plus/minus", "net rating",
    "draw foul rate", 
]

DEFENSIVE_FEATURES = ["steals", "blocks",
                     "opponent's points with player", "defensive rating", "steals to turnovers", "opponent's field goals made",
                      "opponent's field goals attempted", "opp transition shots made", "opp transition shots", 
                      "opp catch and shoot shots", "opp catch and drive shots made", "opp catch and drive shots", 
                      "opp screens off shots made", "opp screens off shots", "opp post up shots made", "opp post up shots",
                      "opp isolations shots made", "opp isolations shots", "opp hand off shots made", "opp hand off shots", 
                      "opp cuts shots made", "opp pick-n-roll shots made", "opp pick-n-roll shots", "opp pick-n-pop shots made", 
                      "opp pick-n-pop shots", "opp drives shots made", "opp drives shots",
]

# Don't need e.g. 'points' because it depends on FGs and FTs made. Same can be said for the following:
# eFG (%), TS (%), FTs (%), 2pt FGs (%), 3pt FGs (%)
# True shooting directly depends on: FT made, FG made, points, FG attempted. Hence, remove these from BASE_FEATURES 
# eFG(%) directly depends on: FG attempted, FG made, 3-pt made. Hence, remove these from BASE_FEATURES
# FG attempted depends on 3-pt attempted. Hence, remove 3-pt attempted from BASE_FEATURES 
BASE_FEATURES = [
    'steals', 'blocks'
]

# Remove these because they are summarized in 'drives made', 'drives with shot', 
# effective field goal percentage, true shooting percentage
NOT_INTERESTING = ["defensive rating", "blocks", "steals", "steals to turnovers"
]

def_data = data_full[DEFENSIVE_FEATURES]
print(f"Number of defensive features: {len(def_data.columns)} out of {len(data_full.columns)}")
def_data_standardized = StandardScaler().fit_transform(def_data)
def_data_standardized_df = pd.DataFrame(def_data_standardized, columns=def_data.columns)


Number of defensive features: 27 out of 93


In [35]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Larger than 6 really deteriorates the performance
MAX_CLUSTERS = 3

# Perform sequential feature selection (on silhouette score) to perform clustering of the offensive data
def sequential_feature_selection(df, use_base_features=False, num_features=20):
    if use_base_features:
        # selected_features = BASE_FEATURES.copy()
        features = [feature for feature in df.columns if feature not in BASE_FEATURES and feature not in NOT_INTERESTING]
    else:
        # selected_features = []
        features = [feature for feature in df.columns if feature not in NOT_INTERESTING]

    silhouette_scores = {}
    final_features = {}
    n_clusters = np.arange(3, MAX_CLUSTERS + 1)
    # For now don't treat the number of clusters as a hyperparameter
    # N_CLUSTERS = 5
    for k in n_clusters:
        cluster_k_scores = []
        if use_base_features:
            cluster_k_features = BASE_FEATURES.copy()
        else:
            cluster_k_features = []
        for i in range(num_features):
            n_features = i + 1
            # Go through all features and calculate silhouette score for each feature. Then select the max.
            curr_silhouette_scores = []
            for feature in features:
                # Append current new feature candidate to the selected features
                curr_features = cluster_k_features + [feature]
                curr_data = def_data_standardized_df[curr_features]
                kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
                kmeans.fit(curr_data)
                score = silhouette_score(curr_data, kmeans.labels_)
                curr_silhouette_scores.append(score)
            
            # Select feature which obtained the maximum silhouette score and add to current features
            max_score = max(curr_silhouette_scores)
            max_score_idx = curr_silhouette_scores.index(max_score)
            # selected_features.append(features[max_score_idx])
            cluster_k_scores.append(max_score)
            cluster_k_features.append(features[max_score_idx])
            if use_base_features:
                print(f"Selected {len(BASE_FEATURES) + n_features}-th feature: {features[max_score_idx]} with silhouette score {max_score} (for cluster size {k})")
            else:
                print(f"Selected {n_features}-th feature: {features[max_score_idx]} with silhouette score {max_score} (for cluster size {k})")
            # Remove max feature from list of features which can be selected
            features.remove(features[max_score_idx])
        silhouette_scores[k] = cluster_k_scores
        final_features[k] = cluster_k_features
        # Reset features for next cluster size
        if use_base_features:
            features = [feature for feature in def_data.columns if feature not in BASE_FEATURES and feature not in NOT_INTERESTING]
        else:
            features = [feature for feature in def_data.columns if feature not in NOT_INTERESTING]

    silhouette_scores_df = pd.DataFrame(silhouette_scores, columns=n_clusters)
    final_features_df = pd.DataFrame(final_features, columns=n_clusters)
    return silhouette_scores_df, final_features_df

silhouette_scores_df, final_features_df = sequential_feature_selection(def_data, use_base_features=False, num_features=20)

Selected 1-th feature: opp pick-n-pop shots made with silhouette score 0.5769624463843248 (for cluster size 3)
Selected 2-th feature: opp pick-n-pop shots with silhouette score 0.4349498942106686 (for cluster size 3)
Selected 3-th feature: opp pick-n-roll shots with silhouette score 0.36898464167492706 (for cluster size 3)
Selected 4-th feature: opp pick-n-roll shots made with silhouette score 0.3509473322389958 (for cluster size 3)
Selected 5-th feature: opp cuts shots made with silhouette score 0.29952234487522705 (for cluster size 3)
Selected 6-th feature: opp post up shots with silhouette score 0.261996411257037 (for cluster size 3)
Selected 7-th feature: opp post up shots made with silhouette score 0.236661823842915 (for cluster size 3)
Selected 8-th feature: opp catch and shoot shots with silhouette score 0.23715207292057652 (for cluster size 3)
Selected 9-th feature: opp catch and drive shots with silhouette score 0.22415251486389676 (for cluster size 3)
Selected 10-th feature: 

Now try also the SFS algorithm if some "base features" are included. 

In [ ]:
silhouette_scores_base_df, final_features_base_df = sequential_feature_selection(def_data, use_base_features=True, num_features=20 - len(BASE_FEATURES))

We look at the different dataframes for each cluster size and choose one which by "eye" has interesting features. Ideally, we would like to have a silhouette score of at least about 0.3. Therefore, we don't consider features which are significantly below that threshold.

While the one without base features has slightly better performance (score: 0.356 vs 0.31), the selected features are highly overlapping. The features it selects which are not selected by the base features versions are:
* '2-pt field goal, %', 'cuts made', 'free throw, %', and 'field goal, %'

We use both as an inspiration to create the following feature set of size 21.

In [36]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

N_CLUSTERS = 2
SELECTED_FEATURES = ["opp cuts shots made", "opp pick-n-roll shots", "opp pick-n-roll shots made", "opp post up shots",
                     "opp post up shots made", "opponent's field goals made", "opponent's field goals attempted",
                     "opp pick-n-pop shots", "opp pick-n-pop shots made", "opp screens off shots", "opp screens off shots made",
                     "opp isolations shots", "opp isolations shots made"
]

# Create the final clustering which can be used in successive steps, e.g. UMAP and t-SNE
final_cluster = KMeans(n_clusters=N_CLUSTERS, init='k-means++', random_state=42, n_init=100)
final_cluster.fit(def_data_standardized_df[SELECTED_FEATURES])
final_score = silhouette_score(def_data_standardized_df[SELECTED_FEATURES], final_cluster.labels_)
print(f"Final silhouette score for selected features: {final_score} for {len(SELECTED_FEATURES)} features")

Final silhouette score for selected features: 0.34377532534041877 for 13 features


Executing the below code for UMAP and t-SNE takes a while. About 15 minutes on my laptop. However, after some good parameters have been found it executes quite fast because we don't have to compute data for 200+ plots.

In [31]:
import umap
from sklearn.manifold import TSNE
# According to ChatGPT it is advisable to use t-SNE and UMAP on the reduced dataset, i.e. containing the selected features
# Hence, we will use the final_features_base_df to reduce the dataset and then apply t-SNE and UMAP to visualize the data
umap_features = SELECTED_FEATURES
print(f"Selected features for UMAP and t-SNE: {umap_features}")
umap_data = def_data_standardized_df[umap_features]

# Apply t-SNE and UMAP to visualize the data
umap_embeddings = {}
tsne_embeddings = {}
# By filtering for offensive features we will also remove the cluster labeling.
# without_cluster = data_full[OFFENSIVE_FEATURES]

# Plot UMAP
# After testing, for min_d > 1 the algorithm returns an error ("divide by zero")
min_distances = [0.1, 0.25, 0.5, 0.8, 1, 1.5, 2, 3]

# for min_d in min_distances:
#     min_d_embedding = []
#     for i in range(20, 201, 20):
#         reducer = umap.UMAP(n_neighbors=i, min_dist=min_d, spread=3)
#         scaled_data = StandardScaler().fit_transform(umap_data)
#         embedding = reducer.fit_transform(scaled_data)
#         min_d_embedding.append(embedding)
#         print(f"finished for n_neighbors = {i} and min_dist = {min_d}")
#     umap_embeddings[min_d] = min_d_embedding

# Plot t-SNE
# for lr in range(40, 200, 10):
#     lr_embedding = []
#     for i in range(5, 51, 5):
#         # Allow for multi-core processing by setting n_jobs=-1
#         tsne = TSNE(n_components=2, perplexity=i, n_iter=300, n_jobs=-1, learning_rate=lr)
#         scaled_data = StandardScaler().fit_transform(umap_data)
#         embedding = tsne.fit_transform(scaled_data)
#         lr_embedding.append(embedding)
#         print(f"finished for perplexity = {i} and learning_rate = {lr}")
#     tsne_embeddings[lr] = lr_embedding

# num_umap_emb = len(umap_embeddings) * len(umap_embeddings[0.1])
# print(f"Num plots for UMAP: {num_umap_emb}")
# num_tsne_emb = len(tsne_embeddings) * len(tsne_embeddings[40])
# print(f"Num plots for t-SNE: {num_tsne_emb}")

Selected features for UMAP and t-SNE: ['opp cuts shots made', 'opp pick-n-roll shots', 'opp pick-n-roll shots made', 'opp post up shots', 'opp post up shots made', "opponent's field goals made", "opponent's field goals attempted", 'opp pick-n-pop shots', 'opp pick-n-pop shots made', 'opp screens off shots', 'opp screens off shots made', 'opp isolations shots', 'opp isolations shots made']


In [ ]:
def plot_tsne(lr, tsne_embeddings):
    fig, axs = plt.subplots(nrows=4, ncols=3, figsize=(30, 20))
    for i, ax in enumerate(axs.flatten()):
        perplexity = (i * 5) + 5
        if i <= 9:
            ax.scatter(tsne_embeddings[i][:, 0], tsne_embeddings[i][:, 1], c=final_cluster.labels_, cmap="Spectral", s=5)
            ax.set_title(f"perplexity: {perplexity} and learning_rate: {lr}")

    plt.tight_layout()
    # Create legend to indicate which cluster has been assigned which color in plot
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='C' + str(i), markersize=10, label=f'Cluster {i}') for i in range(5)]
    plt.legend(handles=handles, title='Clusters', loc='upper right')
    plt.show()

for lr in range(40, 200, 20):
    plot_tsne(lr, tsne_embeddings[lr])

In [ ]:
def plot_umap(min_d, umap_embeddings):
    fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(30, 14))

    for i, ax in enumerate(axs.flatten()):
        neighbours = (i * 20) + 20
        if i <= 5:
            ax.scatter(umap_embeddings[i][:, 0], umap_embeddings[i][:, 1], c=final_cluster.labels_, cmap="plasma", s=5)
            ax.set_title(f"n_neighbors: {neighbours} and min_dist: {min_d}")
        
    plt.tight_layout()
    # Create legend to indicate which cluster has been assigned which color in plot
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='C' + str(i), markersize=10, label=f'Cluster {i}') for i in range(5)]
    plt.legend(handles=handles, title='Clusters', loc='upper right')
    plt.show()

for min_d in min_distances:
    plot_umap(min_d, umap_embeddings[min_d])

In [32]:
# Optionally, add cluster labels back to your original DataFrame
# If you want to attach to the original data, do this. Will also attach to off_data because off_data is only a "view"
# of the full dataframe called data_full.
file = INTERMEDIATE_DATA_DIR / "preprocessed_data.csv"
data_full = pd.read_csv(file)
data_full['cluster'] = final_cluster.labels_.copy()
NORMALIZER = False
# off_stats_reduced['cluster'] = clusters 

# Create a DataFrame for the reduced stats with cluster labels
def analyze_cluster_csv(df, cluster, estimator):
    """
    Assumes that df_reduced (by e.g. PCA) and df have the cluster labels attached.
    """
    # Columns in data frame corresponds to the principal components
    # reduced_df = pd.DataFrame(df_reduced, columns=[f'PC{i+1}' for i in range(df_reduced.shape[1])])
    # reduced_df['cluster'] = cluster
    # # Plotting the mean of each PC for each cluster
    # cluster_means = reduced_df.groupby('cluster').mean()
    # cluster_means.plot(kind='bar', figsize=(14, 7))
    # plt.title('Average Scores for Each Principal Component by Cluster')
    # plt.ylabel('Average Score')
    # plt.xlabel('Cluster')
    # plt.show()

    out_file = CLUSTERS_DIR / f"{str(estimator)}_cluster_stats_def.csv"
    with open(out_file, 'w') as f:
        f.write("")

    # The below code will reomve the cluster column (bc not in OFFENSIVE_FEATURES) from the data frame. Thus, add it back.
    def_data = df[DEFENSIVE_FEATURES].copy()
    def_data['cluster'] = cluster

    # Process each cluster
    for c in range(estimator.n_clusters):
        # Select data for the current cluster
        # print(off_data['cluster'].head())
        cluster_data = def_data[def_data['cluster'] == c]
        # Get the description
        description = cluster_data.describe()
        
        # Write/append the description to a CSV file
        with open(out_file, 'a') as f:
            f.write(f"\nCluster {c}:\n")
            description.to_csv(f)

    print(f"Cluster summaries written to {out_file}")
    
analyze_cluster_csv(data_full, final_cluster.labels_, final_cluster)

Cluster summaries written to KMeans(n_clusters=2, n_init=100, random_state=42)_cluster_stats_def.csv


If needed: analyze the selected features on correlation. If there are many strong (anti)correlations present, apply PCA to further reduce dimensionality.


For our approach, we start by choosing the number of clusters to be 3 and proceed as follows:
1. Analze each cluster for features which distinguishes them from the other two clusters
2. Choose e.g. 5 of them and rank them according to their importance for the distinguishing criteria
3. Come up with three metrics (for three different offensive capabilities) and score players on these for the hexagon plot.

In [38]:
players_df = pd.read_csv(REFERENCE_DATA_DIR / "player_names.csv")
players_df['def_cluster'] = final_cluster.labels_
# Change cluster inedx: 0 to A, 1 to B, 2 to C, etc.
players_df['def_cluster'] = players_df['def_cluster'].apply(lambda x: chr(x + 65))
out_file = CLUSTERS_DIR / "player_cluster_def.csv"

# Write to existing file "player_cluster_memberships.csv" a new column "off_cluster" with the cluster memberships
players_df.to_csv(out_file, index=False)

In [46]:
# For testing:
off_cluster = pd.read_csv(CLUSTERS_DIR / "player_cluster_off.csv")
def_cluster = pd.read_csv(CLUSTERS_DIR / "player_cluster_def.csv")
cluster_data = off_cluster.copy()
cluster_data['def_cluster'] = def_cluster['def_cluster']
print(cluster_data)

             player name  off_cluster def_cluster
0          Pierre Oriola            0           B
1     Usman Garuba Alari            0           B
2          Rolands Smits            0           A
3         Nikola Kalinic            0           A
4             Adam Hanga            1           A
...                  ...          ...         ...
3477   Nikola Tanaskovic            0           B
3478       Arnas Velicka            2           A
3479      Rihards Lomazs            2           A
3480    Rasid Mahalbasic            0           B
3481          Edgar Sosa            2           A

[3482 rows x 3 columns]
